In [422]:
import pandas as pd
import glob
import os

# 🔹 top-level folder (your screenshot root)
root = r"D:\Simmone_MOE_df"

# 🔹 grab everything
files = glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)

print("Total files found:", len(files))

# preview
for f in files[:10]:
    print(f)

Total files found: 101
D:\Simmone_MOE_df\master_boris_post.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\sociability_cohort_1_post_ablation\soc_s1_1_cm1_4_nE_post.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\sociability_cohort_1_post_ablation\soc_s1_1_lfC_nD_post.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\sociability_cohort_1_post_ablation\soc_s1_2_cm1_1_nE_post.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\sociability_cohort_1_post_ablation\soc_s1_2_lfF_nB_post.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\sociability_cohort_1_post_ablation\soc_s1_3_cm1_2_nF_post.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\sociability_cohort_1_post_ablation\soc_s1_3_lfB_nD_VT_CH1_POST.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\sociability_cohort_1_post_ablation\soc_s1_4_cm1_3_nC_VT_CH1_POST.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\sociability_cohort_1_post_ablation\soc_s1_4_lfB_nF_VT_CH1_POST.csv
D:\Simmone_MOE_df\post_ablation_cohort_1_boris\socia

In [423]:
all_behaviors = []
file_info = []

for f in files:
    try:
        df = pd.read_csv(f)
        df.columns = df.columns.str.strip()
        
        if "Behavior" not in df.columns:
            continue
        
        # keep valid rows
        df = df[df["Behavior"].notna()]
        
        # store behaviors
        behaviors = df["Behavior"].unique()
        all_behaviors.extend(behaviors)
        
        # store metadata

        folder = os.path.basename(os.path.dirname(f)).lower()

        if "cohort_1" in folder:
            cohort = "C1"
        elif "cohort_2" in folder:
            cohort = "C2"
        else:
            cohort = "unknown"

        file_info.append({
            "file": os.path.basename(f),
            "folder": folder,
            "cohort": cohort,
            "n_rows": len(df),
            "n_behaviors": len(behaviors)
        })
        
    
    except Exception as e:
        print("Error:", f)

In [424]:
file_df = pd.DataFrame(file_info)

print(file_df.head())

print("\nCohort counts:")
print(file_df["cohort"].value_counts())

print("\nFolders:")
print(file_df["folder"].unique())

                         file                              folder cohort  \
0  soc_s1_1_cm1_4_nE_post.csv  sociability_cohort_1_post_ablation     C1   
1    soc_s1_1_lfC_nD_post.csv  sociability_cohort_1_post_ablation     C1   
2  soc_s1_2_cm1_1_nE_post.csv  sociability_cohort_1_post_ablation     C1   
3    soc_s1_2_lfF_nB_post.csv  sociability_cohort_1_post_ablation     C1   
4  soc_s1_3_cm1_2_nF_post.csv  sociability_cohort_1_post_ablation     C1   

   n_rows  n_behaviors  
0      44            6  
1      33            5  
2      49            6  
3       9            3  
4      68            4  

Cohort counts:
cohort
C1         56
C2         22
unknown    22
Name: count, dtype: int64

Folders:
['sociability_cohort_1_post_ablation'
 'social_memory_cohort_1_post_ablation'
 'sociability_cohort_2_post_ablation'
 'social_memory_cohot_2_post_ablation'
 'sociability_pre_ablation_cohort_1_boris'
 'social memory_pre_ablation_cohort_1_boris'
 'sociability_pre_ablation_cohort_2_boris'
 'soci

In [425]:
print(file_df["cohort"].value_counts())

cohort
C1         56
C2         22
unknown    22
Name: count, dtype: int64


In [426]:
unique_behaviors = sorted(set(all_behaviors))

print("All unique behaviors across dataset:\n")
for b in unique_behaviors:
    print(b)

All unique behaviors across dataset:

Cagemate ROI
Cagemate WC
Climbing on empty cup
Empty Cup ROI
Empty Cup WC
Low fam ROI
Low fam WC
Novel Cup ROI
Novel Cup WC
climbing on cagemate cup
climbing on ledge
climbing on low fam cup
climbing on novel cup


In [427]:
from collections import Counter

behavior_counts = Counter(all_behaviors)

print("\nTop behaviors:\n")
for k, v in behavior_counts.most_common():
    print(f"{k}: {v}")


Top behaviors:

Cagemate WC: 51
Empty Cup WC: 50
Cagemate ROI: 50
Empty Cup ROI: 48
Novel Cup WC: 48
Low fam WC: 47
Novel Cup ROI: 47
Low fam ROI: 46
climbing on ledge: 30
Climbing on empty cup: 27
climbing on cagemate cup: 22
climbing on low fam cup: 22
climbing on novel cup: 21


In [428]:
from collections import defaultdict

behavior_duration = defaultdict(float)

for f in files:
    try:
        df = pd.read_csv(f)
        df.columns = df.columns.str.strip()

        if "Behavior" not in df.columns:
            continue

        df = df[df["Behavior"].notna()].copy()
        df["Duration (s)"] = pd.to_numeric(df["Duration (s)"], errors="coerce")

        for _, row in df.iterrows():
            behavior_duration[row["Behavior"]] += row["Duration (s)"]

    except Exception as e:
        print("Error:", f)

In [429]:
behavior_df = pd.DataFrame([
    {"Behavior": k, "TotalDuration": v}
    for k, v in behavior_duration.items()
])

behavior_df = behavior_df.sort_values("TotalDuration", ascending=False)

behavior_df

,Behavior,TotalDuration
3,Cagemate WC,4023.796
0,Empty Cup WC,3536.692
10,Novel Cup WC,3469.761
11,Novel Cup ROI,3028.712
7,Low fam WC,3008.528
1,Cagemate ROI,2806.090
6,Low fam ROI,2546.239
2,Empty Cup ROI,2089.560
9,climbing on ledge,749.343
5,Climbing on empty cup,400.868


In [430]:
from collections import defaultdict

behavior_duration_pre = defaultdict(float)
behavior_duration_post = defaultdict(float)

for f in files:
    try:
        df = pd.read_csv(f)
        df.columns = df.columns.str.strip()

        if "Behavior" not in df.columns:
            continue

        df = df[df["Behavior"].notna()].copy()
        df["Duration (s)"] = pd.to_numeric(df["Duration (s)"], errors="coerce")

        # 🔥 detect pre/post from folder
        folder = os.path.dirname(f).lower()

        if "pre" in folder:
            target = behavior_duration_pre
        elif "post" in folder:
            target = behavior_duration_post
        else:
            continue

        # 🔹 accumulate durations
        for _, row in df.iterrows():
            target[row["Behavior"]] += row["Duration (s)"]

    except Exception as e:
        print("Error:", f)

In [431]:
all_behaviors_set = set(behavior_duration_pre.keys()) | set(behavior_duration_post.keys())

print("\nBehavior comparison (pre vs post):\n")

for b in sorted(all_behaviors_set):
    pre_val = behavior_duration_pre.get(b, 0)
    post_val = behavior_duration_post.get(b, 0)
    
    print(f"{b:25}  pre: {pre_val:8.2f}   post: {post_val:8.2f}")


Behavior comparison (pre vs post):

Cagemate ROI               pre:  1297.01   post:  1509.08
Cagemate WC                pre:  1151.56   post:  2872.24
Climbing on empty cup      pre:   280.59   post:   120.28
Empty Cup ROI              pre:   594.05   post:  1495.51
Empty Cup WC               pre:   948.03   post:  2588.66
Low fam ROI                pre:   840.25   post:  1705.99
Low fam WC                 pre:  1117.85   post:  1890.68
Novel Cup ROI              pre:  1539.83   post:  1488.88
Novel Cup WC               pre:  1017.32   post:  2452.44
climbing on cagemate cup   pre:   108.26   post:   153.18
climbing on ledge          pre:   140.78   post:   608.56
climbing on low fam cup    pre:   104.71   post:   276.06
climbing on novel cup      pre:   124.04   post:   222.13


In [432]:
print("SOC_pre animals:", SOC_pre["Animal"].nunique())
print("SOC_post animals:", SOC_post["Animal"].nunique())

print("SM_pre animals:", SM_pre["Animal"].nunique())
print("SM_post animals:", SM_post["Animal"].nunique())

SOC_pre animals: 12
SOC_post animals: 16
SM_pre animals: 12
SM_post animals: 16


In [433]:
file_df = pd.DataFrame(file_info)

print(file_df.head())
print("\nTotal files processed:", len(file_df))

                         file                              folder cohort  \
0  soc_s1_1_cm1_4_nE_post.csv  sociability_cohort_1_post_ablation     C1   
1    soc_s1_1_lfC_nD_post.csv  sociability_cohort_1_post_ablation     C1   
2  soc_s1_2_cm1_1_nE_post.csv  sociability_cohort_1_post_ablation     C1   
3    soc_s1_2_lfF_nB_post.csv  sociability_cohort_1_post_ablation     C1   
4  soc_s1_3_cm1_2_nF_post.csv  sociability_cohort_1_post_ablation     C1   

   n_rows  n_behaviors  
0      44            6  
1      33            5  
2      49            6  
3       9            3  
4      68            4  

Total files processed: 100


In [434]:
print("\nAll unique behaviors across dataset:\n")

for b in unique_behaviors:
    print(repr(b))


All unique behaviors across dataset:

'Cagemate ROI'
'Cagemate WC'
'Climbing on empty cup'
'Empty Cup ROI'
'Empty Cup WC'
'Low fam ROI'
'Low fam WC'
'Novel Cup ROI'
'Novel Cup WC'
'climbing on cagemate cup'
'climbing on ledge'
'climbing on low fam cup'
'climbing on novel cup'


In [435]:
import pandas as pd
import os

master_data = []

for f in files:
    try:
        df = pd.read_csv(f)

        # 🔹 clean column names
        df.columns = df.columns.str.strip()

        # 🔥 check Behavior column exists
        if "Behavior" not in df.columns:
            print(f"Skipping (no Behavior column): {f}")
            continue

        # 🔹 keep valid rows
        df = df[df["Behavior"].notna()].copy()

        # 🔹 clean text
        df["Behavior"] = df["Behavior"].astype(str).str.strip()

        # 🔥 remove climbing
        df = df[~df["Behavior"].str.contains("climbing", case=False, na=False)]

        # 🔥 ensure duration column exists
        if "Duration (s)" not in df.columns:
            print(f"Skipping (no Duration column): {f}")
            continue

        df["Duration (s)"] = pd.to_numeric(df["Duration (s)"], errors="coerce")

        # 🔹 summarize behaviors
        summary = df.groupby("Behavior")["Duration (s)"].sum()
        row = summary.to_dict()

        # 🔥 FIX: total session time
        row["Whole"] = df["Duration (s)"].sum()

        # =========================
        # 🔹 METADATA
        # =========================

        fname = os.path.basename(f)
        folder_full = os.path.dirname(f).lower()

        # cohort
        if "cohort_1" in folder_full:
            cohort = "C1"
        elif "cohort_2" in folder_full or "cohot_2" in folder_full:
            cohort = "C2"
        else:
            cohort = "UNK"

        # subject
        parts = fname.split("_")
        if len(parts) < 3:
            print(f"Skipping (bad filename): {f}")
            continue

        cage = parts[1]
        mouse = parts[2]
        subject = f"{cohort}_{cage}.{mouse}"

        # phase
        phase = "soc" if fname.startswith("soc") else "sm"

        # condition
        condition = "pre" if "pre" in folder_full else "post"

        # trial
        if "_cm" in fname:
            trial = "cm"
        elif "_lf" in fname:
            trial = "lf"
        else:
            trial = "other"

        # 🔹 store
        row["Animal"] = subject
        row["Phase"] = phase
        row["Condition"] = condition
        row["Trial"] = trial

        master_data.append(row)

    except Exception as e:
        print(f"Error processing {f}: {e}")


# =========================
# 🔥 BUILD MASTER DF
# =========================

master_df = pd.DataFrame(master_data)

# fill missing behaviors with 0
master_df = master_df.fillna(0)

print(master_df.shape)
master_df.head()

Skipping (no Behavior column): D:\Simmone_MOE_df\master_boris_post.csv
(100, 13)


,Cagemate ROI,Cagemate WC,Empty Cup ROI,Empty Cup WC,Whole,Animal,Phase,Condition,Trial,Low fam ROI,Low fam WC,Novel Cup ROI,Novel Cup WC
0,47.993,44.999,119.243,60.506,272.741,C1_s1.1,soc,post,cm,0.000,0.000,0.0,0.0
1,0.000,0.000,23.007,147.497,253.511,C1_s1.1,soc,post,lf,37.755,45.252,0.0,0.0
2,9.507,58.736,32.758,170.952,271.953,C1_s1.2,soc,post,cm,0.000,0.000,0.0,0.0
3,0.000,0.000,92.263,188.491,299.250,C1_s1.2,soc,post,lf,0.000,18.496,0.0,0.0
4,49.196,193.528,3.254,22.506,268.484,C1_s1.3,soc,post,cm,0.000,0.000,0.0,0.0


In [436]:
# =========================
# 🔹 SOC — CAGEMATE (CM)
# =========================

SOC_CM_pre_C1 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C1"))
].copy()

SOC_CM_post_C1 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C1"))
].copy()


SOC_CM_pre_C2 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C2"))
].copy()

SOC_CM_post_C2 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C2"))
].copy()


# =========================
# 🔹 SOC — LOW FAM (LF)
# =========================

SOC_LF_pre_C1 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "lf") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C1"))
].copy()

SOC_LF_post_C1 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "lf") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C1"))
].copy()


SOC_LF_pre_C2 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "lf") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C2"))
].copy()

SOC_LF_post_C2 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "lf") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C2"))
].copy()

In [437]:
SOC_CM_pre_C1 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C1"))
].copy()

SOC_CM_post_C1 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C1"))
].copy()


SOC_CM_pre_C2 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C2"))
].copy()

SOC_CM_post_C2 = master_df[
    (master_df["Phase"] == "soc") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C2"))
].copy()

In [438]:
print("SOC_CM_pre_C1:", SOC_CM_pre_C1.shape)
print("SOC_CM_post_C1:", SOC_CM_post_C1.shape)
print("SOC_CM_pre_C2:", SOC_CM_pre_C2.shape)
print("SOC_CM_post_C2:", SOC_CM_post_C2.shape)

print("SOC_LF_pre_C1:", SOC_LF_pre_C1.shape)
print("SOC_LF_post_C1:", SOC_LF_post_C1.shape)
print("SOC_LF_pre_C2:", SOC_LF_pre_C2.shape)
print("SOC_LF_post_C2:", SOC_LF_post_C2.shape)

SOC_CM_pre_C1: (6, 13)
SOC_CM_post_C1: (8, 13)
SOC_CM_pre_C2: (4, 13)
SOC_CM_post_C2: (8, 13)
SOC_LF_pre_C1: (6, 13)
SOC_LF_post_C1: (8, 13)
SOC_LF_pre_C2: (2, 13)
SOC_LF_post_C2: (8, 13)


In [439]:
# =========================
# 🔹 SM — LOW FAM (LF)
# =========================

SM_LF_pre_C1 = master_df[
    (master_df["Phase"] == "sm") &
    (master_df["Trial"] == "lf") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C1"))
].copy()

SM_LF_post_C1 = master_df[
    (master_df["Phase"] == "sm") &
    (master_df["Trial"] == "lf") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C1"))
].copy()


SM_LF_pre_C2 = master_df[
    (master_df["Phase"] == "sm") &
    (master_df["Trial"] == "lf") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C2"))
].copy()

SM_LF_post_C2 = master_df[
    (master_df["Phase"] == "sm") &
    (master_df["Trial"] == "lf") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C2"))
].copy()


# =========================
# 🔹 SM — CAGEMATE (CM)
# =========================

SM_CM_pre_C1 = master_df[
    (master_df["Phase"] == "sm") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C1"))
].copy()

SM_CM_post_C1 = master_df[
    (master_df["Phase"] == "sm") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C1"))
].copy()


SM_CM_pre_C2 = master_df[
    (master_df["Phase"] == "sm") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "pre") &
    (master_df["Animal"].str.startswith("C2"))
].copy()

SM_CM_post_C2 = master_df[
    (master_df["Phase"] == "sm") &
    (master_df["Trial"] == "cm") &
    (master_df["Condition"] == "post") &
    (master_df["Animal"].str.startswith("C2"))
].copy()

In [440]:
print("SM_LF_pre_C1:", SM_LF_pre_C1.shape)
print("SM_LF_post_C1:", SM_LF_post_C1.shape)
print("SM_LF_pre_C2:", SM_LF_pre_C2.shape)
print("SM_LF_post_C2:", SM_LF_post_C2.shape)

print("SM_CM_pre_C1:", SM_CM_pre_C1.shape)
print("SM_CM_post_C1:", SM_CM_post_C1.shape)
print("SM_CM_pre_C2:", SM_CM_pre_C2.shape)
print("SM_CM_post_C2:", SM_CM_post_C2.shape)

SM_LF_pre_C1: (6, 13)
SM_LF_post_C1: (8, 13)
SM_LF_pre_C2: (2, 13)
SM_LF_post_C2: (8, 13)
SM_CM_pre_C1: (6, 13)
SM_CM_post_C1: (8, 13)
SM_CM_pre_C2: (4, 13)
SM_CM_post_C2: (8, 13)


In [441]:
print(SOC_CM_pre_C1.shape)
print(SOC_CM_post_C1.shape)
print(SOC_CM_pre_C2.shape)
print(SOC_CM_post_C2.shape)

(6, 13)
(8, 13)
(4, 13)
(8, 13)


In [442]:
SOC_clean = SOC_CM_post_C1.copy()

SOC_clean = SOC_clean.rename(columns={
    "Cagemate ROI": "Social ROI",
    "Empty Cup ROI": "Empty ROI",
    "Cagemate WC": "Social WC",
    "Empty Cup WC": "Empty WC"
})

In [443]:
SOC_clean["Treatment"] = "cagemate"

In [444]:
SOC_clean = SOC_clean[[
    "Animal",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",
    "Social WC",
    "Empty WC"
]]

In [445]:
SOC_clean

,Animal,Treatment,Social ROI,Empty ROI,Whole,Social WC,Empty WC
0,C1_s1.1,cagemate,47.993,119.243,272.741,44.999,60.506
2,C1_s1.2,cagemate,9.507,32.758,271.953,58.736,170.952
4,C1_s1.3,cagemate,49.196,3.254,268.484,193.528,22.506
6,C1_s1.4,cagemate,35.055,25.526,211.398,82.574,68.243
8,C1_s2.1,cagemate,63.460,0.000,285.733,208.401,13.872
10,C1_s2.2,cagemate,196.159,0.000,230.271,30.320,3.792
12,C1_s2.3,cagemate,37.920,8.080,157.249,30.754,80.495
14,C1_s2.4,cagemate,115.280,94.544,284.479,17.664,56.991


In [446]:
SOC_CM_post_C1_clean = SOC_CM_post_C1.copy()

SOC_CM_post_C1_clean = SOC_CM_post_C1_clean.rename(columns={
    "Cagemate ROI": "Social ROI",
    "Empty Cup ROI": "Empty ROI",
    "Cagemate WC": "Social WC",
    "Empty Cup WC": "Empty WC"
})

# Treatment
SOC_CM_post_C1_clean["Treatment"] = "cagemate"

# Fix animal ID
SOC_CM_post_C1_clean["Animal"] = SOC_CM_post_C1_clean["Animal"].str.replace("C1_s", "", regex=False)

# Reorder
SOC_CM_post_C1_clean = SOC_CM_post_C1_clean[[
    "Animal",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",
    "Social WC",
    "Empty WC"
]]

SOC_CM_post_C1_clean.head()

,Animal,Treatment,Social ROI,Empty ROI,Whole,Social WC,Empty WC
0,1.1,cagemate,47.993,119.243,272.741,44.999,60.506
2,1.2,cagemate,9.507,32.758,271.953,58.736,170.952
4,1.3,cagemate,49.196,3.254,268.484,193.528,22.506
6,1.4,cagemate,35.055,25.526,211.398,82.574,68.243
8,2.1,cagemate,63.460,0.000,285.733,208.401,13.872


In [447]:
SOC_CM_post_C2_clean = SOC_CM_post_C2.copy()

SOC_CM_post_C2_clean = SOC_CM_post_C2_clean.rename(columns={
    "Cagemate ROI": "Social ROI",
    "Empty Cup ROI": "Empty ROI",
    "Cagemate WC": "Social WC",
    "Empty Cup WC": "Empty WC"
})

# Treatment
SOC_CM_post_C2_clean["Treatment"] = "cagemate"

# Fix animal ID (remove C2_s prefix)
SOC_CM_post_C2_clean["Animal"] = SOC_CM_post_C2_clean["Animal"].str.replace("C2_s", "", regex=False)


# Reorder columns
SOC_CM_post_C2_clean = SOC_CM_post_C2_clean[[
    "Animal",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",
    "Social WC",
    "Empty WC"
]]


SOC_CM_post_C2_clean.head()

,Animal,Treatment,Social ROI,Empty ROI,Whole,Social WC,Empty WC
32,1.1,cagemate,0.499,44.244,257.488,8.492,204.253
34,1.2,cagemate,9.494,5.248,298.510,281.263,2.505
36,1.3,cagemate,22.675,21.033,237.236,136.178,57.350
38,1.4,cagemate,14.649,22.419,225.095,117.785,70.242
40,2.1,cagemate,47.616,122.544,240.209,10.336,59.713


In [448]:
SOC_LF_post_C1_clean = SOC_LF_post_C1.copy()

SOC_LF_post_C1_clean = SOC_LF_post_C1_clean.rename(columns={
    "Low fam ROI": "Social ROI",
    "Empty Cup ROI": "Empty ROI",
    "Low fam WC": "Social WC",
    "Empty Cup WC": "Empty WC"
})

# Treatment
SOC_LF_post_C1_clean["Treatment"] = "low fam"

# Fix animal ID
SOC_LF_post_C1_clean["Animal"] = SOC_LF_post_C1_clean["Animal"].str.replace("C1_s", "", regex=False)



# Reorder
SOC_LF_post_C1_clean = SOC_LF_post_C1_clean[[
    "Animal",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",
    "Social WC",
    "Empty WC"
]]



SOC_LF_post_C1_clean.head()

,Animal,Treatment,Social ROI,Empty ROI,Whole,Social WC,Empty WC
1,1.1,low fam,37.755,23.007,253.511,45.252,147.497
3,1.2,low fam,0.000,92.263,299.250,18.496,188.491
5,1.3,low fam,9.260,54.896,282.843,10.670,208.017
7,1.4,low fam,47.443,43.950,221.970,64.117,66.460
9,2.1,low fam,15.728,39.824,195.487,5.936,133.999


In [449]:
SOC_LF_post_C1_clean = SOC_LF_post_C1.copy()

SOC_LF_post_C1_clean = SOC_LF_post_C1_clean.rename(columns={
    "Low fam ROI": "Social ROI",
    "Empty Cup ROI": "Empty ROI",
    "Low fam WC": "Social WC",
    "Empty Cup WC": "Empty WC"
})

# Treatment
SOC_LF_post_C1_clean["Treatment"] = "low fam"

# Fix animal ID
SOC_LF_post_C1_clean["Animal"] = SOC_LF_post_C1_clean["Animal"].str.replace("C1_s", "", regex=False)

# 🔥 DO NOT TOUCH Whole — it already exists

# Reorder
SOC_LF_post_C1_clean = SOC_LF_post_C1_clean[[
    "Animal",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",        # ✅ real value from earlier loop
    "Social WC",
    "Empty WC"
]]

SOC_LF_post_C1_clean.head()

,Animal,Treatment,Social ROI,Empty ROI,Whole,Social WC,Empty WC
1,1.1,low fam,37.755,23.007,253.511,45.252,147.497
3,1.2,low fam,0.000,92.263,299.250,18.496,188.491
5,1.3,low fam,9.260,54.896,282.843,10.670,208.017
7,1.4,low fam,47.443,43.950,221.970,64.117,66.460
9,2.1,low fam,15.728,39.824,195.487,5.936,133.999


In [450]:
SOC_LF_post_C2_clean = SOC_LF_post_C2.copy()

SOC_LF_post_C2_clean = SOC_LF_post_C2_clean.rename(columns={
    "Low fam ROI": "Social ROI",
    "Empty Cup ROI": "Empty ROI",
    "Low fam WC": "Social WC",
    "Empty Cup WC": "Empty WC"
})

# Treatment
SOC_LF_post_C2_clean["Treatment"] = "low fam"

# Fix animal ID
SOC_LF_post_C2_clean["Animal"] = SOC_LF_post_C2_clean["Animal"].str.replace("C2_s", "", regex=False)



# Reorder
SOC_LF_post_C2_clean = SOC_LF_post_C2_clean[[
    "Animal",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",
    "Social WC",
    "Empty WC"
]]


SOC_LF_post_C2_clean.head()

,Animal,Treatment,Social ROI,Empty ROI,Whole,Social WC,Empty WC
33,1.1,low fam,6.259,10.996,271.010,27.247,226.508
35,1.2,low fam,2.754,8.750,231.278,37.004,182.770
37,1.3,low fam,45.143,24.640,221.931,98.127,54.021
39,1.4,low fam,96.567,25.224,267.867,100.196,45.880
41,2.1,low fam,59.167,92.240,256.687,25.184,80.096


In [462]:
# =========================
# 🔥 SM LF POST — CLEAN (C1 + C2)
# =========================

# ---- C1 ----
SM_LF_post_C1_clean = SM_LF_post_C1.copy()

SM_LF_post_C1_clean = SM_LF_post_C1_clean.rename(columns={
    "Low fam ROI": "Social ROI",     # familiar
    "Novel Cup ROI": "Novel ROI",
    "Low fam WC": "Social WC",
    "Novel Cup WC": "Novel WC"
})

SM_LF_post_C1_clean["Treatment"] = "low fam"
SM_LF_post_C1_clean["Animal"] = SM_LF_post_C1_clean["Animal"].str.replace("C1_s", "", regex=False)


# ---- C2 ----
SM_LF_post_C2_clean = SM_LF_post_C2.copy()

SM_LF_post_C2_clean = SM_LF_post_C2_clean.rename(columns={
    "Low fam ROI": "Social ROI",
    "Novel Cup ROI": "Novel ROI",
    "Low fam WC": "Social WC",
    "Novel Cup WC": "Novel WC"
})

SM_LF_post_C2_clean["Treatment"] = "low fam"
SM_LF_post_C2_clean["Animal"] = SM_LF_post_C2_clean["Animal"].str.replace("C2_s", "", regex=False)


# =========================
# 🔥 ALIGN + COMBINE
# =========================

cols = [
    "Animal",
    "Treatment",
    "Social ROI",
    "Novel ROI",
    "Whole",
    "Social WC",
    "Novel WC"
]

SM_LF_post_clean = pd.concat([
    SM_LF_post_C1_clean[cols],
    SM_LF_post_C2_clean[cols]
], ignore_index=True).sort_values("Animal").reset_index(drop=True)


# =========================
# 🔥 DONE
# =========================

print(SM_LF_post_clean.shape)
SM_LF_post_clean.head()

(16, 7)


,Animal,Treatment,Social ROI,Novel ROI,Whole,Social WC,Novel WC
0,1.1,low fam,8.249,70.503,253.998,15.999,159.247
1,1.1,low fam,9.996,35.991,239.502,40.253,153.262
2,1.2,low fam,2.002,0.000,300.026,76.012,222.012
3,1.2,low fam,4.994,33.242,258.990,31.499,189.255
4,1.3,low fam,0.000,72.836,297.461,0.000,224.625


In [464]:
# =========================
# 🔥 SM LF POST — CLEAN (FIX ANIMAL IDs)
# =========================

# ---- C1 ----
SM_LF_post_C1_clean = SM_LF_post_C1.copy()

SM_LF_post_C1_clean = SM_LF_post_C1_clean.rename(columns={
    "Low fam ROI": "Social ROI",
    "Novel Cup ROI": "Novel ROI",
    "Low fam WC": "Social WC",
    "Novel Cup WC": "Novel WC"
})

SM_LF_post_C1_clean["Treatment"] = "low fam"

# 🔥 FIX: ADD COHORT TO ANIMAL
SM_LF_post_C1_clean["Animal"] = SM_LF_post_C1_clean["Animal"].astype(str) + "_C1"


# ---- C2 ----
SM_LF_post_C2_clean = SM_LF_post_C2.copy()

SM_LF_post_C2_clean = SM_LF_post_C2_clean.rename(columns={
    "Low fam ROI": "Social ROI",
    "Novel Cup ROI": "Novel ROI",
    "Low fam WC": "Social WC",
    "Novel Cup WC": "Novel WC"
})

SM_LF_post_C2_clean["Treatment"] = "low fam"

# 🔥 FIX: ADD COHORT TO ANIMAL
SM_LF_post_C2_clean["Animal"] = SM_LF_post_C2_clean["Animal"].astype(str) + "_C2"


# =========================
# 🔥 COMBINE
# =========================

cols = [
    "Animal",
    "Treatment",
    "Social ROI",
    "Novel ROI",
    "Whole",
    "Social WC",
    "Novel WC"
]

SM_LF_post_clean = pd.concat([
    SM_LF_post_C1_clean[cols],
    SM_LF_post_C2_clean[cols]
], ignore_index=True).sort_values("Animal").reset_index(drop=True)


print(SM_LF_post_clean.head())

       Animal Treatment  Social ROI  Novel ROI    Whole  Social WC  Novel WC
0  C1_s1.1_C1   low fam       8.249     70.503  253.998     15.999   159.247
1  C1_s1.2_C1   low fam       2.002      0.000  300.026     76.012   222.012
2  C1_s1.3_C1   low fam       0.000     72.836  297.461      0.000   224.625
3  C1_s1.4_C1   low fam      57.669     38.887  234.826     64.425    73.845
4  C1_s2.1_C1   low fam      39.792    121.390  265.649     35.393    69.074


In [463]:
# =========================
# 🔥 SM LF POST — CLEAN (C1 + C2)
# =========================

# ---- C1 ----
SM_LF_post_C1_clean = SM_LF_post_C1.copy()

SM_LF_post_C1_clean = SM_LF_post_C1_clean.rename(columns={
    "Low fam ROI": "Social ROI",     # familiar
    "Novel Cup ROI": "Novel ROI",
    "Low fam WC": "Social WC",
    "Novel Cup WC": "Novel WC"
})

SM_LF_post_C1_clean["Treatment"] = "low fam"
SM_LF_post_C1_clean["Animal"] = SM_LF_post_C1_clean["Animal"].str.replace("C1_s", "", regex=False)


# ---- C2 ----
SM_LF_post_C2_clean = SM_LF_post_C2.copy()

SM_LF_post_C2_clean = SM_LF_post_C2_clean.rename(columns={
    "Low fam ROI": "Social ROI",
    "Novel Cup ROI": "Novel ROI",
    "Low fam WC": "Social WC",
    "Novel Cup WC": "Novel WC"
})

SM_LF_post_C2_clean["Treatment"] = "low fam"
SM_LF_post_C2_clean["Animal"] = SM_LF_post_C2_clean["Animal"].str.replace("C2_s", "", regex=False)


# =========================
# 🔥 ALIGN + COMBINE
# =========================

cols = [
    "Animal",
    "Treatment",
    "Social ROI",
    "Novel ROI",
    "Whole",
    "Social WC",
    "Novel WC"
]

SM_LF_post_clean = pd.concat([
    SM_LF_post_C1_clean[cols],
    SM_LF_post_C2_clean[cols]
], ignore_index=True).sort_values("Animal").reset_index(drop=True)


# =========================
# 🔥 DONE
# =========================

print(SM_LF_post_clean.shape)
SM_LF_post_clean.head()

(16, 7)


,Animal,Treatment,Social ROI,Novel ROI,Whole,Social WC,Novel WC
0,1.1,low fam,8.249,70.503,253.998,15.999,159.247
1,1.1,low fam,9.996,35.991,239.502,40.253,153.262
2,1.2,low fam,2.002,0.000,300.026,76.012,222.012
3,1.2,low fam,4.994,33.242,258.990,31.499,189.255
4,1.3,low fam,0.000,72.836,297.461,0.000,224.625


In [465]:
# =========================
# 🔥 HELPER: FIX ANIMAL ID
# =========================

def fix_animal_id(series, cohort_num):
    return (
        series.astype(str)
        .str.replace("C1_s", "", regex=False)
        .str.replace("C2_s", "", regex=False)
        .str.replace(".", "_", regex=False)
        + f"_{cohort_num}"
    )


# =========================
# 🔥 SM POST (LF + CM)
# =========================

def build_sm(df, treatment, cohort_num):
    df = df.copy()

    if treatment == "low fam":
        df = df.rename(columns={
            "Low fam ROI": "Social ROI",
            "Novel Cup ROI": "Novel ROI",
            "Low fam WC": "Social WC",
            "Novel Cup WC": "Novel WC"
        })
    elif treatment == "cagemate":
        df = df.rename(columns={
            "Cagemate ROI": "Social ROI",
            "Novel Cup ROI": "Novel ROI",
            "Cagemate WC": "Social WC",
            "Novel Cup WC": "Novel WC"
        })

    df["Treatment"] = treatment
    df["Animal"] = fix_animal_id(df["Animal"], cohort_num)

    return df[[
        "Animal","Treatment","Social ROI","Novel ROI","Whole","Social WC","Novel WC"
    ]]


SM_post_clean = pd.concat([
    build_sm(SM_LF_post_C1, "low fam", 1),
    build_sm(SM_LF_post_C2, "low fam", 2),
    build_sm(SM_CM_post_C1, "cagemate", 1),
    build_sm(SM_CM_post_C2, "cagemate", 2)
], ignore_index=True).sort_values(["Animal","Treatment"]).reset_index(drop=True)


# =========================
# 🔥 SOC POST (LF + CM)
# =========================

def build_soc(df, treatment, cohort_num):
    df = df.copy()

    if treatment == "low fam":
        df = df.rename(columns={
            "Low fam ROI": "Social ROI",
            "Empty Cup ROI": "Empty ROI",
            "Low fam WC": "Social WC",
            "Empty Cup WC": "Empty WC"
        })
    elif treatment == "cagemate":
        df = df.rename(columns={
            "Cagemate ROI": "Social ROI",
            "Empty Cup ROI": "Empty ROI",
            "Cagemate WC": "Social WC",
            "Empty Cup WC": "Empty WC"
        })

    df["Treatment"] = treatment
    df["Animal"] = fix_animal_id(df["Animal"], cohort_num)

    return df[[
        "Animal","Treatment","Social ROI","Empty ROI","Whole","Social WC","Empty WC"
    ]]


SOC_post_clean = pd.concat([
    build_soc(SOC_LF_post_C1, "low fam", 1),
    build_soc(SOC_LF_post_C2, "low fam", 2),
    build_soc(SOC_CM_post_C1, "cagemate", 1),
    build_soc(SOC_CM_post_C2, "cagemate", 2)
], ignore_index=True).sort_values(["Animal","Treatment"]).reset_index(drop=True)


# =========================
# 🔥 DONE
# =========================

print("SM:", SM_post_clean.shape)
print("SOC:", SOC_post_clean.shape)

SM_post_clean.head()

SM: (32, 7)
SOC: (32, 7)


,Animal,Treatment,Social ROI,Novel ROI,Whole,Social WC,Novel WC
0,1_1_1,cagemate,172.496,3.245,286.237,99.251,11.245
1,1_1_1,low fam,8.249,70.503,253.998,15.999,159.247
2,1_1_2,cagemate,2.753,121.492,283.499,4.751,154.503
3,1_1_2,low fam,9.996,35.991,239.502,40.253,153.262
4,1_2_1,cagemate,3.000,22.250,165.756,69.748,70.758


In [467]:
# =========================
# 🔥 EXPORT SM
# =========================

SM_post_clean.to_csv(
    r"D:\Simmone_MOE_df\SM_boris_post_clean.csv",
    index=False
)

# =========================
# 🔥 EXPORT SOC
# =========================

SOC_post_clean.to_csv(
    r"D:\Simmone_MOE_df\SOC_boris_post_clean.csv",
    index=False
)

print("Exported SM + SOC successfully")

Exported SM + SOC successfully


In [466]:
# =========================
# 🔥 ADD METADATA + COMBINE
# =========================

# SOC CM
SOC_CM_post_C1_clean["Task"] = "SOC"
SOC_CM_post_C1_clean["Cohort"] = "C1"

SOC_CM_post_C2_clean["Task"] = "SOC"
SOC_CM_post_C2_clean["Cohort"] = "C2"

# SOC LF
SOC_LF_post_C1_clean["Task"] = "SOC"
SOC_LF_post_C1_clean["Cohort"] = "C1"

SOC_LF_post_C2_clean["Task"] = "SOC"
SOC_LF_post_C2_clean["Cohort"] = "C2"

# SM CM
SM_CM_post_C1_clean["Task"] = "SM"
SM_CM_post_C1_clean["Cohort"] = "C1"

SM_CM_post_C2_clean["Task"] = "SM"
SM_CM_post_C2_clean["Cohort"] = "C2"

# SM LF
SM_LF_post_C1_clean["Task"] = "SM"
SM_LF_post_C1_clean["Cohort"] = "C1"

SM_LF_post_C2_clean["Task"] = "SM"
SM_LF_post_C2_clean["Cohort"] = "C2"


# =========================
# 🔥 CONCAT ALL
# =========================

master_boris_post = pd.concat([
    SOC_CM_post_C1_clean,
    SOC_CM_post_C2_clean,
    SOC_LF_post_C1_clean,
    SOC_LF_post_C2_clean,
    SM_CM_post_C1_clean,
    SM_CM_post_C2_clean,
    SM_LF_post_C1_clean,
    SM_LF_post_C2_clean
], ignore_index=True)


# =========================
# 🔥 FINAL FORMAT
# =========================

master_boris_post = master_boris_post[[
    "Animal",
    "Cohort",
    "Task",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",
    "Social WC",
    "Empty WC"
]].sort_values(["Task", "Cohort", "Animal"]).reset_index(drop=True)


# =========================
# 🔥 EXPORT
# =========================

master_boris_post.to_csv(
    r"D:\Simmone_MOE_df\master_boris_post.csv",
    index=False
)

print(master_boris_post.shape)
master_boris_post.head()

(64, 9)


,Animal,Cohort,Task,Treatment,Social ROI,Empty ROI,Whole,Social WC,Empty WC
0,1.1,C1,SM,cagemate,3.245,172.496,286.237,11.245,99.251
1,1.2,C1,SM,cagemate,22.250,3.000,165.756,70.758,69.748
2,1.3,C1,SM,cagemate,1.998,8.198,295.701,26.252,259.253
3,1.4,C1,SM,cagemate,39.359,37.357,172.522,70.670,25.136
4,2.1,C1,SM,cagemate,0.000,87.122,303.104,1.920,214.062


In [458]:
SM_LF_post_C1_clean = SM_LF_post_C1.copy()

SM_LF_post_C1_clean = SM_LF_post_C1_clean.rename(columns={
    "Novel Cup ROI": "Social ROI",   # novel
    "Low fam ROI": "Empty ROI",      # familiar (lf)
    "Novel Cup WC": "Social WC",
    "Low fam WC": "Empty WC"
})

# 🔥 Treatment = familiar identity
SM_LF_post_C1_clean["Treatment"] = "low fam"

SM_LF_post_C1_clean["Animal"] = SM_LF_post_C1_clean["Animal"].str.replace("C1_s", "", regex=False)

# keep Whole (DO NOT overwrite)

SM_LF_post_C1_clean = SM_LF_post_C1_clean[[
    "Animal",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",
    "Social WC",
    "Empty WC"
]]

In [459]:
SM_CM_post_C1_clean = SM_CM_post_C1.copy()

SM_CM_post_C1_clean = SM_CM_post_C1_clean.rename(columns={
    "Novel Cup ROI": "Social ROI",     # novel
    "Cagemate ROI": "Empty ROI",       # familiar (cm)
    "Novel Cup WC": "Social WC",
    "Cagemate WC": "Empty WC"
})

SM_CM_post_C1_clean["Treatment"] = "cagemate"

SM_CM_post_C1_clean["Animal"] = SM_CM_post_C1_clean["Animal"].str.replace("C1_s", "", regex=False)

SM_CM_post_C1_clean = SM_CM_post_C1_clean[[
    "Animal",
    "Treatment",
    "Social ROI",
    "Empty ROI",
    "Whole",
    "Social WC",
    "Empty WC"
]]